# Lab dl_07 · Decoding: temperature, top-k, top-p — and the KV cache

**Goal.** Implement the standard sampling knobs and a KV-cached causal attention layer, and prove that incremental decoding
gives exactly the same result as a full forward pass.

### Theory recap
Given logits $z$: **temperature** $p_i\propto e^{z_i/T}$ ($T<1$ sharpens, $T\to0$ is greedy, $T>1$ flattens).
**Top-$k$** keeps the $k$ largest logits. **Top-$p$ (nucleus)** sorts tokens by probability and keeps the smallest prefix whose
total probability is $\ge p$ (always at least one token), then renormalises. Order used here: temperature → top-k → top-p → softmax → sample.

**KV cache.** In causal attention the keys/values of past tokens never change, so when generating token $t+1$ we only compute
$q,k,v$ for the new token and attend over the cached $K_{1..t}$, $V_{1..t}$ plus the new ones: $O(t)$ work per token instead of
re-running the whole prefix. Memory: $2\cdot n_{layer}\cdot n_{kv}\cdot d_{head}\cdot T\cdot B\cdot\text{bytes}$.
With a cache of $P$ past positions and $T_{new}$ new queries, query $i$ (absolute position $P+i$) may attend to keys $0..P+i$.

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import math
import time
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Sampling filters
All functions take logits of shape (B, V) and return logits of the same shape (filtered entries set to `-inf`).

In [ ]:
def apply_temperature(logits, temperature):
    # TODO
    raise NotImplementedError


def top_k_filter(logits, k):
    # TODO
    raise NotImplementedError


def top_p_filter(logits, p):
    # TODO (hint: sort, softmax, cumsum; keep a token if the mass BEFORE it is < p; scatter back to the original order)
    raise NotImplementedError


def sample_next(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    # TODO: returns ids of shape (B,)
    raise NotImplementedError

In [ ]:
z = torch.tensor([[3.0, 2.5, 2.0, 1.0, 0.5, 0.0, -1.0, -2.0]])
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (name, f) in zip(axes, [("T=1", lambda z: z), ("T=0.5", lambda z: apply_temperature(z, 0.5)),
                                ("top-k=3", lambda z: top_k_filter(z, 3)), ("top-p=0.8", lambda z: top_p_filter(z, 0.8))]):
    ax.bar(range(8), F.softmax(f(z), -1)[0]); ax.set_title(name)
plt.show()

## 2. KV-cached causal self-attention
Attributes `self.qkv = nn.Linear(d, 3d)`, `self.proj = nn.Linear(d, d)`, heads via `.view(B, T, h, d_head).transpose(1, 2)`. `forward(x, kv_cache=None)` returns `(y, (k_all, v_all))` with `k_all`, `v_all` of shape (B, h, T_past + T_new, d_head).

In [ ]:
class CachedCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x, kv_cache=None):
        # TODO
        raise NotImplementedError


def kv_cache_bytes(n_layers, n_kv_heads, d_head, seq_len, batch=1, bytes_per_el=2):
    # TODO
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
attn = CachedCausalSelfAttention(64, 4).eval()
x = torch.randn(1, 50, 64)
with torch.no_grad():
    full, _ = attn(x)
    cache, outs = None, []
    for t in range(50):
        y, cache = attn(x[:, t:t + 1], cache); outs.append(y)
print("max |full - incremental| =", (full - torch.cat(outs, 1)).abs().max().item())

# cost: re-running the whole prefix every step vs. using the cache
with torch.no_grad():
    x = torch.randn(1, 512, 64)
    t0 = time.time()
    for t in range(1, 513): attn(x[:, :t])
    t_nocache = time.time() - t0
    t0 = time.time(); cache = None
    for t in range(512): _, cache = attn(x[:, t:t + 1], cache)
    t_cache = time.time() - t0
print(f"no cache {t_nocache:.2f}s vs cache {t_cache:.2f}s")
print("Llama-2-7B, 4096 tokens, fp16:", kv_cache_bytes(32, 32, 128, 4096) / 2**30, "GiB")

### Reflect (on paper)
1. Why can't the *queries* be cached? Why is the cache useless for a bidirectional (BERT-style) encoder?
2. Grouped-query attention with 8 KV heads instead of 32: how much KV memory does Llama-3-8B save vs. Llama-2-7B at 8k context?
3. With batch 1, decoding is memory-bandwidth bound. Estimate tokens/s for a 7B fp16 model on a 896 GB/s GPU.

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_07_sampling", globals())